# Ch 34 (검증) — 한국어 diffusion, 80/10/10 수정판

순진한 100% [MASK] diffusion은 한국어에서 유니그램 붕괴(acc 0.08). BERT의 80/10/10 마스킹 트릭 + plain CE 이식으로 해결(검증 acc 0.467). 생성까지 확인.

In [1]:
%pip install -q -U transformers tokenizers datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 11.1/11.2 MB 170.7 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 93.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━ 38.8/48.9 MB 141.4 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 48.9/48.9 MB 70.6 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 48.9/48.9 MB 70.6 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 48.9/48.9 MB 70.6 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 12.7 MB/s eta 0:00:00


In [2]:
import math, time, torch
import torch.nn.functional as F
from datasets import load_dataset, Dataset
SEED=42; torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
USE_FP16 = torch.cuda.is_available()
print("device", device, "| fp16", USE_FP16)

device cuda | fp16 True


## 1. 한국어 TinyStories 복원 (Ch 26과 동일)

In [3]:
EOT="<|endoftext|>"; N_TRAIN,N_VAL,MAXL=50_000,500,1_500_000
def rebuild(split,n,maxl):
    stories,buf=[],[]
    for i,ex in enumerate(load_dataset("g0ster/TinyStories-Korean",split=split,streaming=True)):
        if i>=maxl or len(stories)>=n: break
        line=(ex["text"] or "").strip()
        if line==EOT:
            s=" ".join(buf).strip()
            if s: stories.append(s)
            buf=[]
        elif line: buf.append(line)
    if buf and len(stories)<n:
        s=" ".join(buf).strip()
        if s: stories.append(s)
    return stories[:n]
raw_train=Dataset.from_dict({"text":rebuild("train",N_TRAIN,MAXL)})
raw_val=Dataset.from_dict({"text":rebuild("validation",N_VAL,50_000)})
print("stories", len(raw_train), len(raw_val))
print(raw_train[0]["text"][:120])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/789 [00:00<?, ?B/s]

stories 50000 500
한때 벤이라는 이름의 어린 소년이 있었어요. 벤은 주변 세계를 탐험하는 것을 좋아했답니다. 그는 가게에 전시되어 있던 아름다운 꽃병들 같은 멋진 것들을 많이 봤어요. 어느 날, 벤은 가게를 거닐다가 정말 특별한 꽃병


## 2. BPE 4000 + initial_alphabet + [MASK]

In [4]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from transformers import PreTrainedTokenizerFast
VOCAB=4000
def corpus_iter(bs=1000):
    for i in range(0,len(raw_train),bs): yield raw_train[i:i+bs]["text"]
_tk=Tokenizer(models.BPE(unk_token="[UNK]"))
_tk.pre_tokenizer=pre_tokenizers.ByteLevel(add_prefix_space=False)
_tk.decoder=decoders.ByteLevel()
_tk.train_from_iterator(corpus_iter(), trainer=trainers.BpeTrainer(
    vocab_size=VOCAB, special_tokens=["[PAD]","[UNK]","[MASK]"],
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet()))
tokenizer=PreTrainedTokenizerFast(tokenizer_object=_tk, pad_token="[PAD]", unk_token="[UNK]", mask_token="[MASK]")
print("vocab", tokenizer.vocab_size, "mask_id", tokenizer.mask_token_id)

vocab 4000 mask_id 2


## 3. 토큰화 + group

In [5]:
BLOCK=128
tt=raw_train.map(lambda b: tokenizer(b["text"],add_special_tokens=False), batched=True, remove_columns=raw_train.column_names)
tv=raw_val.map(lambda b: tokenizer(b["text"],add_special_tokens=False), batched=True, remove_columns=raw_val.column_names)
def group(b):
    cat=sum(b["input_ids"],[]); n=(len(cat)//BLOCK)*BLOCK
    return {"input_ids":[cat[i:i+BLOCK] for i in range(0,n,BLOCK)]}
lm_train=tt.map(group,batched=True,remove_columns=tt.column_names)
lm_val=tv.map(group,batched=True,remove_columns=tv.column_names)
print("chunks", len(lm_train), len(lm_val))

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

chunks 75941 746


## 4. ★수정 — diffusion 콜레이터에 80/10/10 (순진한 100% [MASK] 대신)

가변 마스킹률 t는 유지(생성용)하되, 선택된 자리에 80% [MASK] / 10% 랜덤 / 10% 원본유지. 이게 모델이 [MASK]→유니그램 지름길로 새는 걸 막는다.

In [6]:
N_SPECIAL=3
class DiffMLMCollator:
    def __init__(self, tok, eps=0.05, tmax=1.0, seed=SEED):
        self.mask_id=tok.mask_token_id; self.vocab=tok.vocab_size
        self.eps=eps; self.tmax=tmax; self.gen=torch.Generator().manual_seed(seed)
    def __call__(self, ex):
        ids=torch.tensor([e["input_ids"] for e in ex], dtype=torch.long)
        B,L=ids.shape
        t=torch.rand(B,generator=self.gen)*(self.tmax-self.eps)+self.eps
        sel=torch.rand(B,L,generator=self.gen)<t.unsqueeze(1)
        no=~sel.any(1)
        if no.any():
            j=torch.randint(0,L,(int(no.sum()),),generator=self.gen); sel[no,j]=True
        labels=ids.clone(); labels[~sel]=-100
        inp=ids.clone()
        r=torch.rand(B,L,generator=self.gen)
        inp[sel&(r<0.8)]=self.mask_id                                    # 80% [MASK]
        rp=sel&(r>=0.8)&(r<0.9); nr=int(rp.sum())                        # 10% 랜덤
        if nr: inp[rp]=torch.randint(N_SPECIAL,self.vocab,(nr,),generator=self.gen)
        # 10% 원본 유지
        return {"input_ids":inp,"attention_mask":torch.ones(B,L,dtype=torch.long),"labels":labels}
coll=DiffMLMCollator(tokenizer)

## 5. 작은 모델 (256/4L) — 용량 아닌 마스킹이 문제였음

In [7]:
from transformers import BertConfig, BertForMaskedLM
cfg=BertConfig(vocab_size=tokenizer.vocab_size, hidden_size=256, num_hidden_layers=4,
               num_attention_heads=4, intermediate_size=1024,
               max_position_embeddings=BLOCK, pad_token_id=tokenizer.pad_token_id)
model=BertForMaskedLM(cfg).to(device)
print("params(M)", round(model.num_parameters()/1e6,2))

params(M) 4.29


## 6. 학습 — plain CE(BertForMaskedLM 기본) + lr 5e-4, 30000 step

In [8]:
from transformers import Trainer, TrainingArguments
args=TrainingArguments(output_dir="./out34", max_steps=30000,
    per_device_train_batch_size=64, learning_rate=5e-4, weight_decay=0.01,
    warmup_steps=1000, lr_scheduler_type="cosine", max_grad_norm=1.0, fp16=USE_FP16,
    logging_steps=500, save_strategy="no", report_to="none", remove_unused_columns=False, seed=SEED)
trainer=Trainer(model=model, args=args, train_dataset=lm_train, data_collator=coll)
t0=time.time(); r=trainer.train()
print(f"elapsed {(time.time()-t0)/60:.2f}min | step {r.global_step} | train_loss {r.training_loss:.4f} | baseline ln(V) {math.log(tokenizer.vocab_size):.4f}")

Step,Training Loss
500,7.181070
1000,6.543603
1500,6.420502
2000,6.362722
2500,6.217342
3000,5.823278
3500,5.186558
4000,4.761226
4500,4.556010
5000,4.405662


elapsed 20.11min | step 30000 | train_loss 4.1110 | baseline ln(V) 8.2940


## 7. carry-over 샘플러로 한국어 생성

In [9]:
@torch.no_grad()
def generate(model, length=128, block=32, temperature=0.8, top_p=0.92, top_k=0,
             rep_penalty=1.3, no_immediate_repeat=True, prompt_ids=None):
    """carry-over semi-AR + 반복 억제(rep penalty / 인접중복 금지 / top-p)."""
    model.eval()
    mask_id = tokenizer.mask_token_id
    x = torch.full((1, length), mask_id, dtype=torch.long, device=device)
    fixed = torch.zeros(length, dtype=torch.bool, device=device)
    if prompt_ids is not None:
        p = torch.tensor(prompt_ids[:length], device=device)
        x[0, :len(p)] = p; fixed[:len(p)] = True
    nblocks = (length + block - 1) // block
    for b in range(nblocks):
        lo, hi = b * block, min((b + 1) * block, length)
        steps = hi - lo
        for s in range(steps):
            logits = model(input_ids=x).logits[0].float()        # (L, V)
            logits[:, mask_id] = -1e9
            # 반복 패널티: 이미 확정된 토큰들의 로짓을 깎음
            if rep_penalty and rep_penalty != 1.0:
                comm = x[0][x[0] != mask_id]
                if comm.numel() > 0:
                    u = torch.unique(comm)
                    col = logits[:, u]
                    logits[:, u] = torch.where(col > 0, col / rep_penalty, col * rep_penalty)
            # 인접중복 금지: 각 자리에서 '왼쪽 토큰과 같은 토큰' 예측 차단
            if no_immediate_repeat:
                left = torch.roll(x[0], 1); left[0] = mask_id
                valid = left != mask_id
                logits[valid, left[valid]] = -1e9
            probs = (logits / max(temperature, 1e-6)).softmax(-1)
            if top_k and top_k > 0:
                kth = probs.topk(top_k, dim=-1).values[:, -1, None]
                probs = probs.masked_fill(probs < kth, 0.0)
            if top_p and top_p < 1.0:
                sp, si = probs.sort(dim=-1, descending=True)
                rm = (sp.cumsum(-1) - sp) > top_p
                sp = sp.masked_fill(rm, 0.0)
                probs = torch.zeros_like(probs).scatter(-1, si, sp)
            probs = probs / probs.sum(-1, keepdim=True).clamp_min(1e-9)
            pred = torch.multinomial(probs, 1).squeeze(-1)
            conf = probs.gather(-1, pred.unsqueeze(-1)).squeeze(-1)
            cur = (x[0] == mask_id) & (~fixed)
            cur[:lo] = False; cur[hi:] = False
            nleft = int(cur.sum())
            if nleft == 0: break
            nreveal = nleft if s == steps - 1 else max(1, nleft // (steps - s))
            cc = conf.clone(); cc[~cur] = -1e9
            idx = cc.topk(nreveal).indices
            x[0, idx] = pred[idx]
    return tokenizer.decode(x[0], skip_special_tokens=True)

pid = tokenizer("옛날 옛날에", add_special_tokens=False)["input_ids"]
torch.manual_seed(SEED)
print("=== unconditional (all-[MASK] -> generate, default sampler) ===")
for i in range(3):
    print(f"[{i}] {generate(model)[:340]}")
print("\n=== conditional (prompt 'Once upon a time' fixed) ===")
for i in range(3):
    print(f"[{i}] {generate(model, prompt_ids=pid)[:340]}")

pid = tokenizer("옛날 옛날에", add_special_tokens=False)["input_ids"]
torch.manual_seed(SEED)
print("=== conditional ('옛날 옛날에') ===")
for i in range(3):
    print(f"[{i}] {generate(model, prompt_ids=pid)[:300]}")
print("\n=== unconditional ===")
for i in range(2):
    print(f"[{i}] {generate(model)[:300]}")

=== unconditional (all-[MASK] -> generate, default sampler) ===


[0] !한때 작은 마을에 팀이라는 어린 소년이 살고 있었습니다. 그는 매우 행복했습니다. 어느 날, 팀은 불꽃이 공원에 갔습니다. 그들은 큰 불꽃을 보았습니다. 그 불꽃은 크고 밝고 반짝거렸습니다. 그것은 반짝이고 예뻤습니다. 팀과 그의 엄마는 함께 놀고 싶어 했습니다. 하지만 그때 뜻밖의 일이 벌어졌습니다. 팀이 불꽃놀이를 보고 말했습니다: "팀아, 불꽃놀이가 정말 예쁘네!"라고 말했습니다. 엄마가 웃으며 대답했습니다, " 불꽃놀이는 행복하게 만들 수 있어." 팀은 동의했고 팀의 엄마는 미소를 지었습니다. 이제 불꽃은 다시 불꽃놀이를 가지게 되었다는 사실에 기뻐했습니다. 그래서 그들은 매일 불꽃과 함께 놀


[1] 옛날 옛적에 팀이라는 작은 소년이 있었어요. 팀은 장난감 가지고 노는 것을 매우 좋아했지요. 어느 날, 그는 바닥에서 가장 좋아하는 장난감 자동차를 발견했어요. 그 자동차는 아주 빠르게 움직일 수 없어서 슬펐답니다. 팀이 놀고 있을 때, 팀의 장난감 자동차가 고장 난 걸 보고 말았어요. 팀은 슬퍼하며 울기 시작했죠. 엄마가 방에 들어오며 "팀아, 내가 네 차를 고쳐줄게!"라고 말했죠. 엄마는 웃으며 말했어요. "괜찮아, 이제 고칠게."라고 말씀하셨어요. 그들은 함께 장난감 자동차로 즐겁게 놀며 정말 즐거운 시간을 보냈답니다.오래전 옛날에 팀이라는 이름의 어린 소년이 있었습니다. 팀은 크고 멋진 장난감을


[2]  어느 날, 공원을 산책하러 산책을 나갔습니다. 걷는 작은 새를 보았습니다. 새가 말했습니다, "안녕, 새야! 나랑 놀고 싶어?" 새는 대답했습니다, "응, 너랑 놀 수 있을까? 나도 같이 가고 싶지 않니?" 톰은 잠시 생각한 뒤에 말했습니다, "그래, 톰!" 그들은 매일 함께 놀았습니다. 해가 지기 시작하자 톰과 그 새는 좋은 친구가 되었습니다.옛날 옛적에 작은 마을에 릴리라는 소녀가 살고 있었습니다. 그녀는 친구들과 노는 것을 아주 좋아했지요. 어느 날, 릴리는 자신의 침대에서 크고 포근한 테디베어를 발견했습니다. 테디베어는 매우 특별하고 행복했습니다. 그래서 릴리는 테디베어와 부드러운 부드러운 테

=== conditional (prompt 'Once upon a time' fixed) ===


[0] 옛날 옛날에, 팀이라는 이름의 이름의 행복한 어린 소년이 살고 있었어요. 팀은 장난감 가지고 노는 것을 정말 좋아했지요. 어느 날, 그는 바닥에 떨어져 있는 새로운 장난감을 발견했어요. 그 장난감은 빨간색의 바퀴가 달린 것이었어요. 팀은 자신의 새 자동차와 놀고 싶어 했어요. 그래서 팀의 엄마는 "자동차! 나랑 같이 놀자!"라고 말했어요. 팀은 자동차를 들고 밖으로 나가 함께 놀았답니다. 하지만 뜻밖의 일이 벌어졌어요. 갑자기 자동차가 큰 방이 움직이기 시작한 거예요! 팀과 그의 엄마가 방에 들어왔어요. 그녀는 팀을 보고 웃었어요. 그들은 매우 기뻐했어요. 둘은 하루 종일 장난감 자동차로 놀며 즐거운 


...

[0] 옛날 옛날에, 작은 집에 팀이라는 이름의 어린 소년이 살고 있었어요. 팀은 장난감 가지고 노는 것을 매우 좋아했죠. 어느 날, 그는 자신의 방에서 큰 상자를 발견했어요. 상자 안에는 많은 장난감과 게임들이 가득 차 있었죠. 팀은 그 상자에 넣어보고 싶어 했죠. 그래서 엄마는 "그래, 한번 열어보자!"라고 했어요. 팀은 엄마에게 도움을 청했어요. 그들은 장난감들을 모두 넣고 상자를 열었어요. 상자는 너무 커서 쉽게 열 수가 없었어요. 하지만 상자가 열리지 않고 포기하지 않았어요. 팀은 그게 무슨 뜻인지 몰랐지만, 그때 예상치 못한 일


[1] 옛날 옛날에, 조용한 마을에 릴리라는 이름의 작은 소녀가 있었어요. 그녀는 친구들과 공원에서 함께 노는 것을 좋아했지요. 어느 날, 릴리는 풀밭에서 큰 나무를 발견했어요. 그 나무는 크고 많은 잎사귀가 있었죠. 릴리는 나무 아래를 내려다보며 꽃을 봤어요. 그 나무가 정말 예쁘다고 생각했답니다. 하지만 그때, 친절한 새가 공원에 왔어요. "왜 그렇게 슬퍼하니 않니?"라고 물었어요. 새는 잠시 생각한 뒤 말했어요. "네가 같이 놀고 싶어요."라고 했어요. 릴리와 그녀의 친구들은 웃으며 재미있게 놀았어요. 해가 쨍쨍 비추고 새들이 지저귀


[2] 옛날 옛날에, 수라는 이름의 작은 소녀가 방에 살고 있었어요. 그녀는 장난감 자동차를 가지고 노는 것을 좋아했죠. 어느 날, 수는 자신의 방에서 큰 상자를 발견했어요. 그녀는 그 자동차와 함께 놀고 싶어 했죠. 하지만 상자는 너무 작아서 움직일지 않았어요. 수가 루시에게 말했어요, "엄마, 저랑 같이 놀 수 있을까요?" 엄마는 대답했어요, "그래, 수야! 나는 내 동생이 있어." 그들은 종일 장난감 자동차로 정말 재미있게 놀았어요. 그런데, 뜻밖의 일이 벌어졌어요. 그녀의 엄마가 장난감을 보고 말을 하기 시작한 거예요! 장난감이 아

=== unconditional ===


[0]  수 있다는 것을 배웠습니다.어느 날, 팀이라는 소년이 엄마와 함께 공원에 갔어요. 그들은 큰 나무를 보고 그 나무에 올라가고 싶었지만 여전히 그네를 타고 싶었어요. 팀은 그네에 있는 작은 새를 보았어요. 팀은 새가 미끄럼틀을 타는 걸 봤어요. 그는 엄마에게 "엄마, 저도 같이 갈 수 있을까요?"라고 물었어요. 엄마는 "그래, 팀아! 하지만 먼저 그네를 다 탈 수 있어." 팀과 그의 엄마는 웃으며 말했어요. "응, 팀! 미끄럼틀 타면서 정말 멋진 시간이야!"이라고 말했죠. 팀은 너무 기뻐서 함께 그네를 타며 높은 새로 내려갔죠. 언


[1]  있는 큰 나무를 보았습니다. 그는 그와 놀고 싶어 했습니다. 팀의 엄마는 웃으며 말했습니다. "걱정 마, 팀! 나는 너를 도와줄게." 팀은 매우 기뻐했습니다. 그는 나무 아래 앉아서 앉아 쉬었습니다. 그들은 하루 종일 함께 놀았습니다. 팀과 그의 엄마는 좋은 친구가 되었습니다.옛날 옛적에 팀이라는 이름의 어린 소년이 있었습니다. 어느 날, 그는 땅에 반짝이는 무언가를 발견했습니다. 그것은 반짝이고 둥근 것이었습니다. 팀은 그것을 주워 들고 가지고 싶었습니다. 그는 그것이 재미있을 것 같다고 생각했습니다. 그래서 그는 돌을 집어 들


## 8. 진단 — 고정-t(0.15) acc + infill

In [10]:
g=torch.Generator().manual_seed(0)
def fixed_t_acc(tv_=0.15,n=128):
    cor=tot=0
    for ex in lm_val.select(range(min(n,len(lm_val)))):
        ids=torch.tensor(ex["input_ids"]); m=torch.rand(len(ids),generator=g)<tv_
        if not m.any(): m[0]=True
        inp=ids.clone(); inp[m]=tokenizer.mask_token_id
        with torch.no_grad(): pr=model(inp.unsqueeze(0).to(device)).logits[0].argmax(-1).cpu()
        cor+=(pr[m]==ids[m]).sum().item(); tot+=int(m.sum())
    return cor/tot
print(f"[diag] fixed-t(0.15) top-1 acc = {fixed_t_acc():.3f}   (naive diffusion 0.084)")

[diag] fixed-t(0.15) top-1 acc = 0.651   (naive diffusion 0.084)
